# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [16]:
import sys
print(sys.executable)

/Users/praxideswanyoike/ai/projects/tinyml-arduino/bin/python


In [17]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [18]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [20]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [21]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
x = wine.data
y = wine.target

print ("Feature shape", x.shape)
print ("Labels shape:", y.shape)



Feature shape (178, 13)
Labels shape: (178,)


In [22]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
x_train, x_test, y_train, y_test = train_test_split (
    x,
    y,
    test_size = .3,
    random_state = 42
)

In [23]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scale = StandardScaler()

x_train = scale.fit_transform(x_train)
x_test = scale.transform(x_test)

In [24]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train = tf.keras.utils.to_categorical(y_train, num_classes = num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes = num_classes)


In [25]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
model = tf.keras.Sequential ([
    tf.keras.layers.Dense(64, activation = 'relu', input_shape = (x_train.shape[1],)),
    tf.keras.layers.Dense(32, activation = 'relu'),
    tf.keras.layers.Dense(num_classes, activation = 'softmax')
])
                          

In [26]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile (
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

history = model.fit(
    x_train, 
    y_train, 
    epochs = 20, 
    batch_size = 8,
    validation_split = .2
)

Epoch 1/20
13/13 [==============================] - 0s 5ms/step - loss: 1.0090 - accuracy: 0.5152 - val_loss: 0.8505 - val_accuracy: 0.6000
Epoch 2/20
13/13 [==============================] - 0s 836us/step - loss: 0.7317 - accuracy: 0.8081 - val_loss: 0.6098 - val_accuracy: 0.8800
Epoch 3/20
13/13 [==============================] - 0s 821us/step - loss: 0.5449 - accuracy: 0.8889 - val_loss: 0.4541 - val_accuracy: 0.9200
Epoch 4/20
13/13 [==============================] - 0s 790us/step - loss: 0.4074 - accuracy: 0.9192 - val_loss: 0.3385 - val_accuracy: 0.9600
Epoch 5/20
13/13 [==============================] - 0s 731us/step - loss: 0.3034 - accuracy: 0.9495 - val_loss: 0.2623 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 737us/step - loss: 0.2297 - accuracy: 0.9697 - val_loss: 0.1960 - val_accuracy: 1.0000
Epoch 7/20
13/13 [==============================] - 0s 829us/step - loss: 0.1768 - accuracy: 0.9798 - val_loss: 0.1509 - val_accuracy: 1.0000
Epoch 8/

In [28]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
test_loss, test_accuracy = model.evaluate(x_test, y_test)

print("Test Accuracy:", test_accuracy)
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis =1)
y_true = np.argmax(y_test, axis = 1)

print ("Classification report:" )
print (classification_report (y_true, y_pred_classes))
print ("confusion matrix:" )
print (confusion_matrix(y_true, y_pred_classes))

2/2 [==============================] - 0s 2ms/step - loss: 0.0885 - accuracy: 0.9815
Test Accuracy: 0.9814814925193787
2/2 [==============================] - 0s 970us/step
Classification report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        19
           1       1.00      1.00      1.00        21
           2       1.00      0.93      0.96        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

confusion matrix:
[[19  0  0]
 [ 0 21  0]
 [ 1  0 13]]


In [29]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open ("model_base.tflite", "wb") as f: 
    f.write(tflite_model)

    print ("Saved model_base.tflite")
    print ("Model size:" , round(len(tflite_model) / 1024,2) , "KB")


INFO:tensorflow:Assets written to: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmptvzo39ab/assets


INFO:tensorflow:Assets written to: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmptvzo39ab/assets


Saved model_base.tflite
Model size: 14.1 KB


2026-05-17 18:39:44.947361: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-17 18:39:44.947402: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-17 18:39:44.948353: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmptvzo39ab
2026-05-17 18:39:44.948640: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-17 18:39:44.948643: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmptvzo39ab
2026-05-17 18:39:44.950846: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-17 18:39:44.951139: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-17 18:39:44.992005: I tensorflow/cc/saved_model/loader.

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [49]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]

def representative_dataset_gen():
    return representative_data_gen(x_train)

def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset_gen
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8
        ]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
  

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    # <-- Enter your code here <--#
    tflite_model = converter.convert()
    with open(filename, "wb") as f: 
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_det = interpreter.get_input_details()
    output_det = interpreter.get_output_details()
    y_pred = []

    for i in range(len(x_test)): 
        sample = x_test[i:i + 1].astype(np.float32)

        if input_det[0]["dtype"] == np.int8:
            scale, zero_point = input_det[0]["quantization"]
            sample = sample / scale + zero_point
            sample = np.round(sample).astype(np.int8)


        interpreter.set_tensor(
            input_det[0]["index"],
            sample
        )

        interpreter.invoke()

        output = interpreter.get_tensor(
            output_det[0]["index"]
        )

        if output_det[0]["dtype"] == np.int8:
            scale, zero_point = output_det[0]["quantization"]

            output = (output.astype(np.float32) - zero_point) * scale

        y_pred.append(np.argmax(output))
            

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis = 1)
    def file_size_kb(filename): 
        with open(filename, "rb") as f: 
            data = f.read()
            return len(data) / 1024

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#
    print ("\nClassifcation Report:")
    print(classification_report(y_true, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


In [50]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
quantize_and_evaluate (
    model, 
    x_test,
    y_test,
    "int8",
    "model_int8.tflite"
)

quantize_and_evaluate (
    model,
    x_test,
    y_test,
    "float16",
    "model_float16.tflite"
)

quantize_and_evaluate (
    model,
    x_test,
    y_test,
    "dynamic",
    "model_dynamic.tlfite"
)


INFO:tensorflow:Assets written to: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmphc8c7mn9/assets


INFO:tensorflow:Assets written to: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmphc8c7mn9/assets



INT8 TFLite model size: 5.76 KB

Classifcation Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        19
           1       1.00      1.00      1.00        21
           2       1.00      0.93      0.96        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 1  0 13]]
INFO:tensorflow:Assets written to: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmp2c93n24c/assets


/Users/praxideswanyoike/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-17 21:22:35.334441: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-17 21:22:35.334454: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-17 21:22:35.334547: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmphc8c7mn9
2026-05-17 21:22:35.334873: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-17 21:22:35.334876: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmphc8c7mn9
2026-05-17 21:22:35.335744: I tensorflow/cc/saved_model/loader


FLOAT16 TFLite model size: 9.00 KB

Classifcation Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        19
           1       1.00      1.00      1.00        21
           2       1.00      0.93      0.96        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 1  0 13]]
INFO:tensorflow:Assets written to: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmp1l_99_30/assets


INFO:tensorflow:Assets written to: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmp1l_99_30/assets



DYNAMIC TFLite model size: 8.20 KB

Classifcation Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        19
           1       1.00      1.00      1.00        21
           2       1.00      0.93      0.96        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 1  0 13]]


2026-05-17 21:22:35.676552: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-17 21:22:35.676561: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-17 21:22:35.676630: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmp1l_99_30
2026-05-17 21:22:35.676918: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-17 21:22:35.676923: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmp1l_99_30
2026-05-17 21:22:35.677738: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-17 21:22:35.689767: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmp1l_99_30
2026-05-

## Problem 1 - Part (c)

### Pruning

In [51]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#

batch_size = 8
epochs = 20

end_step = int(np.ceil(len(x_train) / batch_size) * epochs)

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay (
    initial_sparsity = .5,
    final_sparsity = .7,
    begin_step = 0,
    end_step = end_step
)



In [52]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
prune = tf.keras.Sequential([
    tfmot.sparsity.keras.prune_low_magnitude(
    tf.keras.layers.Dense(
        64,
        activation = "relu",
        input_shape = (x_train.shape[1],)
    ),
    pruning_schedule = pruning_schedule
    ),
    tfmot.sparsity.keras.prune_low_magnitude(
        tf.keras.layers.Dense(
            32,
            activation = "relu"
        ),
    
        pruning_schedule = pruning_schedule
    ),
    tfmot.sparsity.keras.prune_low_magnitude (
        tf.keras.layers.Dense(
            3,
            activation = "softmax"
        ),
        pruning_schedule = pruning_schedule
    )
])
        
                             

In [55]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
prune.compile (
    optimizer = "adam",
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

history = prune.fit (
    x_train,
    y_train,
    epochs = 10,
    batch_size = 8,
    validation_split = .2,
    callbacks =[tfmot.sparsity.keras.UpdatePruningStep()]
)

Epoch 1/10
13/13 [==============================] - 1s 4ms/step - loss: 1.1740 - accuracy: 0.3939 - val_loss: 0.9679 - val_accuracy: 0.6800
Epoch 2/10
13/13 [==============================] - 0s 980us/step - loss: 0.8494 - accuracy: 0.7980 - val_loss: 0.7259 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.6299 - accuracy: 0.8990 - val_loss: 0.5628 - val_accuracy: 0.8800
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.4676 - accuracy: 0.9495 - val_loss: 0.4387 - val_accuracy: 0.8800
Epoch 5/10
13/13 [==============================] - 0s 933us/step - loss: 0.3436 - accuracy: 0.9596 - val_loss: 0.3490 - val_accuracy: 0.8800
Epoch 6/10
13/13 [==============================] - 0s 948us/step - loss: 0.2527 - accuracy: 0.9596 - val_loss: 0.2908 - val_accuracy: 0.8800
Epoch 7/10
13/13 [==============================] - 0s 981us/step - loss: 0.1888 - accuracy: 0.9697 - val_loss: 0.2426 - val_accuracy: 0.9200
Epoch 8/10
1

In [57]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
stripped_model = tfmot.sparsity.keras.strip_pruning(prune)
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_model = converter.convert()
with open ("model_pruned.tflite", "wb") as f: 
    f.write(tflite_model)

print("Saved model_pruned.tflite")
print("Model size:", round(len(tflite_model) / 1024, 2), "KB")


INFO:tensorflow:Assets written to: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmpbkr9d9l9/assets


INFO:tensorflow:Assets written to: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmpbkr9d9l9/assets


Saved model_pruned.tflite
Model size: 14.16 KB


2026-05-17 21:48:08.387683: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-17 21:48:08.387696: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-17 21:48:08.387774: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmpbkr9d9l9
2026-05-17 21:48:08.387971: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-17 21:48:08.387973: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmpbkr9d9l9
2026-05-17 21:48:08.388398: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-17 21:48:08.393251: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmpbkr9d9l9
2026-05-

In [59]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred = stripped_model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis = 1)
y_true = np.argmax(y_test, axis = 1)

print ("\nClassification Report:")
print(classification_report(y_true, y_pred_classes))
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred_classes))

2/2 [==============================] - 0s 1ms/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        19
           1       0.95      1.00      0.98        21
           2       1.00      1.00      1.00        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[18  1  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [65]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
student_mod = tf.keras.Sequential([
    tf.keras.layers.Dense(
        32, 
        activation = "relu",
        input_shape = (x_train.shape[1],)
    ),
    tf.keras.layers.Dense(
        16, 
        activation="relu"
    ),
    tf.keras.layers.Dense(
        3,
        activation = "softmax"
    )
])

In [66]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
teach_soft_label = model.predict(x_train)

4/4 [==============================] - 0s 799us/step


In [69]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
teach_pred_soft = model.predict(x_train)
y_train_combine = np.concatenate(
    [y_train, teach_pred_soft],
    axis=1
)


def distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    alpha = .5
    hard_loss = tf.keras.losses.categorical_crossentropy(
        y_true_hard,
        y_pred
    )

    hard_loss = tf.keras.losses.categorical_crossentropy (
        y_true_hard,
        y_pred
    )

    soft_loss = tf.keras.losses.categorical_crossentropy(
        y_true_soft, 
        y_pred
    )
    return alpha * hard_loss + (1 - alpha) * soft_loss

    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    pass

4/4 [==============================] - 0s 785us/step


In [70]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
student_mod.compile (
    optimizer = "adam",
    loss = distillation_loss, 
    metrics=["accuracy"]
)

history = student_mod.fit (
    x_train,
    y_train_combine,
    epochs = 10,
    batch_size = 8, 
    validation_split = .2
)

Epoch 1/10
13/13 [==============================] - 0s 4ms/step - loss: 1.4370 - accuracy: 0.2727 - val_loss: 1.2344 - val_accuracy: 0.2800
Epoch 2/10
13/13 [==============================] - 0s 856us/step - loss: 1.1608 - accuracy: 0.3333 - val_loss: 1.0229 - val_accuracy: 0.3200
Epoch 3/10
13/13 [==============================] - 0s 816us/step - loss: 0.9802 - accuracy: 0.3939 - val_loss: 0.8851 - val_accuracy: 0.4000
Epoch 4/10
13/13 [==============================] - 0s 778us/step - loss: 0.8524 - accuracy: 0.5556 - val_loss: 0.7753 - val_accuracy: 0.5600
Epoch 5/10
13/13 [==============================] - 0s 732us/step - loss: 0.7489 - accuracy: 0.6970 - val_loss: 0.6879 - val_accuracy: 0.8000
Epoch 6/10
13/13 [==============================] - 0s 726us/step - loss: 0.6662 - accuracy: 0.7576 - val_loss: 0.6087 - val_accuracy: 0.8800
Epoch 7/10
13/13 [==============================] - 0s 726us/step - loss: 0.5858 - accuracy: 0.8182 - val_loss: 0.5371 - val_accuracy: 0.8800
Epoch 8/

In [71]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(student_mod)

tflite_model = converter.convert()
with open ("model_kd.tflite", "wb") as f: 
    f.write(tflite_model)

print("Saved model_kd.tflite")
print("Model size:", round(len(tflite_model) / 1024, 2), "KB")

INFO:tensorflow:Assets written to: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmpv86f8w82/assets


INFO:tensorflow:Assets written to: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmpv86f8w82/assets


Saved model_kd.tflite
Model size: 6.14 KB


2026-05-17 22:22:09.919080: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-17 22:22:09.919091: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-17 22:22:09.919180: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmpv86f8w82
2026-05-17 22:22:09.919515: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-17 22:22:09.919518: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmpv86f8w82
2026-05-17 22:22:09.920374: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-17 22:22:09.932318: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/mj/y1x2tmfn7nz93rgy_fnv55ym0000gp/T/tmpv86f8w82
2026-05-

In [72]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred = student_mod.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis = 1)
y_true = np.argmax (y_test, axis =1)

print("\nClassifcation report:")
print(classification_report(y_true, y_pred_classes))
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred_classes))

2/2 [==============================] - 0s 1ms/step

Classifcation report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      0.95      0.98        21
           2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [ ]:
# <-- (if needed) Enter your code here <--#

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
